# 03 Feature Engineering Audit

Leakage spot-check notebook. This verifies that behavioral features use only events before each offer `received_time`.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data/processed'
FEATURES = pd.read_parquet(DATA_DIR / 'features.parquet')
TRANSCRIPT = pd.read_parquet(DATA_DIR / 'transcript_flat.parquet')
RESPONSE = pd.read_parquet(DATA_DIR / 'offer_response.parquet')


In [ ]:
sample_rows = (
    FEATURES.loc[FEATURES['n_transactions_before'] > 0, [
        'person',
        'offer_id',
        'received_time',
        'n_transactions_before',
        'total_spend_before',
        'avg_spend_before',
        'offers_received_before',
        'offers_completed_before',
        'offer_completion_rate_before',
    ]]
    .sample(5, random_state=42)
    .sort_values(['person', 'received_time'])
    .reset_index(drop=True)
)
sample_rows


In [ ]:
audit_rows = []

for row in sample_rows.itertuples(index=False):
    person_transcript = TRANSCRIPT.loc[
        (TRANSCRIPT['person'] == row.person) & (TRANSCRIPT['time'] < row.received_time)
    ].copy()
    prior_transactions = person_transcript.loc[person_transcript['event'] == 'transaction'].copy()
    prior_transactions['amount'] = pd.to_numeric(prior_transactions['amount'], errors='coerce').fillna(0.0)

    prior_offers = RESPONSE.loc[
        (RESPONSE['person'] == row.person) & (RESPONSE['received_time'] < row.received_time)
    ].copy()

    actual_n_transactions = int(len(prior_transactions))
    actual_total_spend = float(prior_transactions['amount'].sum())
    actual_avg_spend = float(prior_transactions['amount'].mean()) if actual_n_transactions else 0.0
    actual_received = int(len(prior_offers))
    actual_completed = int(prior_offers['label'].sum()) if actual_received else 0
    actual_completion_rate = (actual_completed / actual_received) if actual_received else 0.0

    audit_rows.append({
        'person': row.person,
        'offer_id': row.offer_id,
        'received_time': row.received_time,
        'feature_n_transactions_before': int(row.n_transactions_before),
        'actual_n_transactions_before': actual_n_transactions,
        'feature_total_spend_before': float(row.total_spend_before),
        'actual_total_spend_before': actual_total_spend,
        'feature_avg_spend_before': float(row.avg_spend_before),
        'actual_avg_spend_before': actual_avg_spend,
        'feature_offers_received_before': int(row.offers_received_before),
        'actual_offers_received_before': actual_received,
        'feature_offers_completed_before': int(row.offers_completed_before),
        'actual_offers_completed_before': actual_completed,
        'feature_offer_completion_rate_before': float(row.offer_completion_rate_before),
        'actual_offer_completion_rate_before': actual_completion_rate,
    })

audit_df = pd.DataFrame(audit_rows)
audit_df


In [ ]:
comparison_columns = [
    ('feature_n_transactions_before', 'actual_n_transactions_before'),
    ('feature_total_spend_before', 'actual_total_spend_before'),
    ('feature_avg_spend_before', 'actual_avg_spend_before'),
    ('feature_offers_received_before', 'actual_offers_received_before'),
    ('feature_offers_completed_before', 'actual_offers_completed_before'),
    ('feature_offer_completion_rate_before', 'actual_offer_completion_rate_before'),
]

results = audit_df[['person', 'offer_id', 'received_time']].copy()
for left_col, right_col in comparison_columns:
    results[f'{left_col}_matches'] = (audit_df[left_col].round(6) == audit_df[right_col].round(6))

results['all_checks_pass'] = results.drop(columns=['person', 'offer_id', 'received_time']).all(axis=1)
results
